# Interview Orchestrator

This notebook implements the interview state machine and demonstrates
the orchestration logic used at runtime by `orchestrator.py`.

**States:** `INIT → PROFILING → QUESTIONING → EVALUATING → MEMORY_UPDATE → ENDED`

## Cell 1 — Setup & Imports

In [ ]:
import sys, os, enum, json, uuid, logging
from typing import Any, Dict
from uuid import UUID

# Ensure project root is on the path for imports
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..', '..', '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger('interview_orchestrator')
print(f'Project root: {PROJECT_ROOT}')

## Cell 2 — State Machine Definition

In [ ]:
class InterviewState(str, enum.Enum):
    """States in the interview state machine."""
    INIT = 'init'
    PROFILING = 'profiling'
    QUESTIONING = 'questioning'
    EVALUATING = 'evaluating'
    MEMORY_UPDATE = 'memory_update'
    ENDED = 'ended'

print('States defined:', [s.value for s in InterviewState])

## Cell 3 — Agent Imports

In [ ]:
from app.agents.interview.sub_agents.profile_intelligence.agent import analyze_profile
from app.agents.interview.sub_agents.question_generator.agent import generate_question
from app.agents.interview.sub_agents.answer_evaluator.agent import evaluate_answer
from app.agents.interview.sub_agents.memory_agent.agent import update_memory
from app.agents.interview.sub_agents.feedback_agent.agent import generate_report
from app.agents.interview.sub_agents.speech_to_text.agent import transcribe
from app.agents.interview.sub_agents.text_to_speech.agent import synthesize
from app.agents.interview.llm_provider import get_llm

print('All 7 sub-agents + LLM provider imported successfully.')

## Cell 4 — Orchestrator Class

In [ ]:
class InterviewOrchestrator:
    """
    Stateful orchestrator that manages the flow of an interview session.
    Each call to step() determines the next agent to invoke.
    """

    def __init__(self, student_id: UUID, session_id: UUID, db, llm) -> None:
        self.student_id = student_id
        self.session_id = session_id
        self.db = db
        self.llm = llm
        self.state = InterviewState.INIT
        self._profile_context = ''
        self._current_difficulty = 'medium'
        self._question_count = 0
        self._max_questions = 15

    async def step(self, last_answer: str = '') -> Dict[str, Any]:
        logger.info('Orchestrator step: state=%s', self.state.value)

        # INIT → PROFILING → first QUESTION
        if self.state == InterviewState.INIT:
            self.state = InterviewState.PROFILING
            profile_data = await analyze_profile(self.student_id, self.db, self.llm)
            skills_str = ', '.join(profile_data.get('skills', []))
            domains_str = ', '.join(profile_data.get('domains', []))
            exp = profile_data.get('experience_level', 'junior')
            self._profile_context = (
                f'Candidate skills: {skills_str}. '
                f'Domains: {domains_str}. '
                f'Experience level: {exp}.'
            )
            self.state = InterviewState.QUESTIONING
            question_data = await generate_question(
                context=self._profile_context,
                difficulty=self._current_difficulty,
                llm=self.llm,
            )
            self._question_count += 1
            return {
                'next_agent': 'question_generator',
                'difficulty': question_data.get('difficulty', self._current_difficulty),
                'action': 'ask_question',
                'data': {'profile': profile_data, 'question': question_data},
            }

        # QUESTIONING → generate next question
        if self.state == InterviewState.QUESTIONING:
            if self._question_count >= self._max_questions:
                return await self._end_interview()
            question_data = await generate_question(
                context=self._profile_context,
                difficulty=self._current_difficulty,
                llm=self.llm,
            )
            self._question_count += 1
            return {
                'next_agent': 'question_generator',
                'difficulty': question_data.get('difficulty', self._current_difficulty),
                'action': 'ask_question',
                'data': {'question': question_data},
            }

        # EVALUATING → evaluate + update memory
        if self.state == InterviewState.EVALUATING:
            eval_data = await evaluate_answer(
                question=last_answer, answer=last_answer, llm=self.llm,
            )
            self.state = InterviewState.MEMORY_UPDATE
            memory_data = await update_memory(
                session_id=self.session_id, answer=last_answer,
                db=self.db, llm=self.llm,
            )
            self._current_difficulty = eval_data.get('next_difficulty', 'medium')
            if self._question_count >= self._max_questions:
                return await self._end_interview()
            self.state = InterviewState.QUESTIONING
            return {
                'next_agent': 'answer_evaluator',
                'difficulty': self._current_difficulty,
                'action': 'ask_question',
                'data': {'evaluation': eval_data, 'memory': memory_data},
            }

        # ENDED → generate report
        if self.state == InterviewState.ENDED:
            report = await generate_report(
                session_id=self.session_id, db=self.db, llm=self.llm,
            )
            return {
                'next_agent': 'feedback_agent',
                'difficulty': self._current_difficulty,
                'action': 'end',
                'data': {'report': report},
            }

        return {'next_agent': 'none', 'difficulty': self._current_difficulty, 'action': 'end', 'data': {}}

    async def _end_interview(self) -> Dict[str, Any]:
        self.state = InterviewState.ENDED
        report = await generate_report(
            session_id=self.session_id, db=self.db, llm=self.llm,
        )
        return {
            'next_agent': 'feedback_agent',
            'difficulty': self._current_difficulty,
            'action': 'end',
            'data': {'report': report},
        }

print('InterviewOrchestrator class defined.')

## Cell 5 — State Transition Diagram

In [ ]:
transitions = {
    'INIT':          '→ PROFILING → QUESTIONING (auto)',
    'QUESTIONING':   '→ (waits for answer submission)',
    'EVALUATING':    '→ MEMORY_UPDATE → QUESTIONING | ENDED',
    'MEMORY_UPDATE': '→ QUESTIONING (if questions remain)',
    'ENDED':         '→ generates report via FeedbackAgent',
}

print('=== Interview State Transitions ===')
for state, desc in transitions.items():
    print(f'  {state:15s} {desc}')

## Cell 6 — Example Run (requires DB + Gemini key)

In [ ]:
# Uncomment to run a live test:
#
# import asyncio
# from app.core.database import async_session_factory
#
# async def demo():
#     llm = get_llm()
#     async with async_session_factory() as db:
#         orch = InterviewOrchestrator(
#             student_id=uuid.uuid4(),
#             session_id=uuid.uuid4(),
#             db=db,
#             llm=llm,
#         )
#         result = await orch.step()
#         print(json.dumps(result, indent=2, default=str))
#
# asyncio.run(demo())

print('Notebook ready. Uncomment Cell 6 to run a live demo.')